In [6]:
!pip install rapidfuzz


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t', nrows=2)
print("Columns in Source 1:", s1.columns.tolist())

Columns in Source 1: ['entity_id', 'business_name', 'business_address', 'country']


In [8]:
import pandas as pd
gt = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t', nrows=2)
print("Columns in Ground Truth:", gt.columns.tolist())

Columns in Ground Truth: ['source1_entity_id', 'matched_entity_ids']


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import fuzz
from tqdm import tqdm

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import fuzz
from tqdm import tqdm

print("=== AMAZON ML CHALLENGE 2026: FEATURE ENGINEERING ===")

# 1. Load the TRAIN candidates
print("1. Loading candidate pairs...")
candidates = pd.read_csv("../output/candidate_pairs_train.tsv", sep='\t')

candidates['candidate_entity_ids'] = candidates['candidate_entity_ids'].astype(str).str.split(',')
df_pairs = candidates.explode('candidate_entity_ids').replace('', np.nan).dropna()
df_pairs = df_pairs.rename(columns={'candidate_entity_ids': 'candidate_id'})

print(f"Total candidate pairs to evaluate: {len(df_pairs):,}")

# 2. Load Raw Data for Text Lookups
print("2. Loading raw sources from student_resource...")
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t') 
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')

s2_s3_combined = pd.concat([s2, s3], ignore_index=True)

s1[['business_name', 'business_address']] = s1[['business_name', 'business_address']].fillna('')
s2_s3_combined[['business_name', 'business_address']] = s2_s3_combined[['business_name', 'business_address']].fillna('')

# 3. Merge Text onto the Pairs
print("3. Merging text data onto pairs...")
df_pairs = df_pairs.merge(
    s1[['entity_id', 'business_name', 'business_address']].rename(columns={
        'business_name': 's1_name', 'business_address': 's1_address'
    }),
    left_on='source1_entity_id', right_on='entity_id', how='left'
).drop(columns=['entity_id'])

df_pairs = df_pairs.merge(
    s2_s3_combined[['entity_id', 'business_name', 'business_address']].rename(columns={
        'business_name': 'cand_name', 'business_address': 'cand_address'
    }),
    left_on='candidate_id', right_on='entity_id', how='left'
).drop(columns=['entity_id'])

# 4. Generate Target Labels
print("4. Attaching Ground Truth labels...")
ground_truth = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t')

ground_truth['matched_entity_ids'] = ground_truth['matched_entity_ids'].astype(str).str.split(',')
gt_exploded = ground_truth.explode('matched_entity_ids').replace('', np.nan).dropna()
gt_exploded = gt_exploded.rename(columns={'matched_entity_ids': 'candidate_id'})
gt_exploded['is_match'] = 1

df_pairs = df_pairs.merge(
    gt_exploded[['source1_entity_id', 'candidate_id', 'is_match']],
    on=['source1_entity_id', 'candidate_id'],
    how='left'
)
df_pairs['label'] = df_pairs['is_match'].fillna(0).astype(int)
df_pairs = df_pairs.drop(columns=['is_match'])

print(f"Label distribution:\n{df_pairs['label'].value_counts(normalize=True) * 100}")

# 5. Compute Similarity Features
print("5. Computing string similarity features...")
s1_names = df_pairs['s1_name'].tolist()
cand_names = df_pairs['cand_name'].tolist()
s1_addrs = df_pairs['s1_address'].tolist()
cand_addrs = df_pairs['cand_address'].tolist()

df_pairs['name_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names), desc="Name Fuzz")]
df_pairs['name_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names), desc="Name Token Sort")]
df_pairs['addr_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs), desc="Address Fuzz")]
df_pairs['addr_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs), desc="Address Token Sort")]

# 6. Boolean & Categorical Features
print("6. Calculating boolean features...")
df_pairs['exact_name_match'] = (df_pairs['s1_name'].str.lower() == df_pairs['cand_name'].str.lower()).astype(int)
df_pairs['s1_missing_addr'] = (df_pairs['s1_address'] == '').astype(int)
df_pairs['cand_missing_addr'] = (df_pairs['cand_address'] == '').astype(int)
df_pairs['both_missing_addr'] = (df_pairs['s1_missing_addr'] & df_pairs['cand_missing_addr']).astype(int)

features_to_save = df_pairs.drop(columns=['s1_name', 'cand_name', 's1_address', 'cand_address'])

# 7. Save Dataset
output_file = Path("../output/training_features.parquet")
features_to_save.to_parquet(output_file, index=False)
print(f"\nSuccessfully saved {len(features_to_save):,} feature rows to {output_file}")

=== AMAZON ML CHALLENGE 2026: FEATURE ENGINEERING ===
1. Loading candidate pairs...
Total candidate pairs to evaluate: 65,993,761
2. Loading raw sources from student_resource...


just checking score

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, precision_score, recall_score

print("1. Loading only 'name_fuzz_ratio' and 'label' (~500 MB RAM)...")
# Specifying columns bypasses loading the heavy string IDs
df = pd.read_parquet(
    "../output/training_features.parquet", 
    columns=['name_fuzz_ratio', 'label']
)

print(f"Loaded {len(df):,} rows successfully.")

# 2. Split 20% for validation
print("2. Splitting validation set...")
_, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

y_val = val_df['label'].values
ratios = val_df['name_fuzz_ratio'].values

# Free memory from the full dataframe
del df

# 3. Evaluate F0.5 across different similarity cutoffs
print("\n=== BASELINE EVALUATION (Rule-based Name Fuzz) ===")
print(f"{'Threshold':<10} | {'F0.5 Score':<12} | {'Precision':<10} | {'Recall':<10}")
print("-" * 50)

for cutoff in [70, 75, 80, 85, 90, 95]:
    preds = (ratios >= cutoff).astype(int)
    
    f05 = fbeta_score(y_val, preds, beta=0.5, zero_division=0)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    
    print(f">= {cutoff:<7} | {f05:<12.4f} | {prec:<10.4f} | {rec:<10.4f}")

1. Loading only 'name_fuzz_ratio' and 'label' (~500 MB RAM)...
Loaded 65,993,761 rows successfully.
2. Splitting validation set...

=== BASELINE EVALUATION (Rule-based Name Fuzz) ===
Threshold  | F0.5 Score   | Precision  | Recall    
--------------------------------------------------
>= 70      | 0.5478       | 0.5251     | 0.6625    
>= 75      | 0.5675       | 0.5588     | 0.6049    
>= 80      | 0.5841       | 0.5967     | 0.5383    
>= 85      | 0.6010       | 0.6572     | 0.4477    
>= 90      | 0.5890       | 0.7311     | 0.3314    
>= 95      | 0.4662       | 0.7919     | 0.1762    


soo i got a f 0.5 score of .85 so i am now featurn scalimng and after that i will do hyperparameter tuning
if there is enough time 

In [ ]:
# import pandas as pd
# import numpy as np
# from rapidfuzz import fuzz
# from tqdm import tqdm
# import re

# print("=== AMAZON ML CHALLENGE 2026: PRO FEATURE ENGINEERING ===")

# # --- 1. Text Normalization Function ---
# def normalize_text(text):
#     if not isinstance(text, str):
#         return ''
#     text = text.lower()
#     # Strip global corporate suffixes
#     suffixes = [
#         r'\bprivate\b', r'\blimited\b', r'\bpvt\b', r'\bltd\b',
#         r'\bllc\b', r'\binc\b', r'\bcorp\b', r'\bcorporation\b',
#         r'\band\b', r'\bco\b', r'\bcompany\b'
#     ]
#     for suffix in suffixes:
#         text = re.sub(suffix, '', text)
#     # Remove punctuation
#     text = re.sub(r'[^\w\s]', '', text)
#     return ' '.join(text.split()) # Clean up extra spaces

# # --- 2. Load Data ---
# print("1. Loading candidate pairs...")
# candidates = pd.read_csv("../output/candidate_pairs_train.tsv", sep='\t')
# candidates['candidate_entity_ids'] = candidates['candidate_entity_ids'].astype(str).str.split(',')
# df_pairs = candidates.explode('candidate_entity_ids').replace('', np.nan).dropna()
# df_pairs = df_pairs.rename(columns={'candidate_entity_ids': 'candidate_id'})
# print(f"Total candidate pairs: {len(df_pairs):,}")

# print("\n2. Loading raw sources...")
# s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
# s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
# s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')

# s2_s3_combined = pd.concat([s2, s3], ignore_index=True)
# s1[['business_name', 'business_address']] = s1[['business_name', 'business_address']].fillna('')
# s2_s3_combined[['business_name', 'business_address']] = s2_s3_combined[['business_name', 'business_address']].fillna('')

# # --- 3. Clean Text BEFORE Merging (Huge Speed Boost) ---
# print("\n3. Normalizing business names...")
# s1['business_name_clean'] = s1['business_name'].apply(normalize_text)
# s2_s3_combined['business_name_clean'] = s2_s3_combined['business_name'].apply(normalize_text)

# # --- 4. Merge Text Data ---
# print("\n4. Merging clean text data onto pairs...")
# df_pairs = df_pairs.merge(
#     s1[['entity_id', 'business_name_clean', 'business_address']].rename(columns={
#         'business_name_clean': 's1_name', 'business_address': 's1_address'
#     }),
#     left_on='source1_entity_id', right_on='entity_id', how='left'
# ).drop(columns=['entity_id'])

# df_pairs = df_pairs.merge(
#     s2_s3_combined[['entity_id', 'business_name_clean', 'business_address']].rename(columns={
#         'business_name_clean': 'cand_name', 'business_address': 'cand_address'
#     }),
#     left_on='candidate_id', right_on='entity_id', how='left'
# ).drop(columns=['entity_id'])

# # --- 5. Ground Truth Labels ---
# print("\n5. Attaching Ground Truth labels...")
# gt = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
# gt_set = set(zip(gt['source1_entity_id'].astype(str), gt['source2_entity_id'].astype(str)))
# df_pairs['label'] = df_pairs.apply(
#     lambda row: 1 if (str(row['source1_entity_id']), str(row['candidate_id'])) in gt_set else 0,
#     axis=1
# ).astype(np.uint8)

# # --- 6. Similarity Features ---
# print("\n6. Computing similarity features on clean names...")
# s1_names, cand_names = df_pairs['s1_name'].tolist(), df_pairs['cand_name'].tolist()
# s1_addrs, cand_addrs = df_pairs['s1_address'].tolist(), df_pairs['cand_address'].tolist()

# df_pairs['name_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names))]
# df_pairs['name_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names))]
# df_pairs['addr_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs))]
# df_pairs['addr_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs))]

# df_pairs['exact_name_match'] = (df_pairs['s1_name'] == df_pairs['cand_name']).astype(np.uint8)
# df_pairs['s1_missing_addr'] = (df_pairs['s1_address'] == '').astype(np.uint8)
# df_pairs['cand_missing_addr'] = (df_pairs['cand_address'] == '').astype(np.uint8)
# df_pairs['both_missing_addr'] = (df_pairs['s1_missing_addr'] & df_pairs['cand_missing_addr']).astype(np.uint8)

# # --- 7. Save Dataset ---
# print("\n7. Saving clean training features...")
# feature_cols = [
#     'source1_entity_id', 'candidate_id', 
#     'name_fuzz_ratio', 'name_token_sort', 'addr_fuzz_ratio', 'addr_token_sort',
#     'exact_name_match', 's1_missing_addr', 'cand_missing_addr', 'both_missing_addr',
#     'label'
# ]

# df_pairs[feature_cols].to_parquet("../output/training_features.parquet", index=False)
# print("Successfully overwrote training_features.parquet with clean text features!")

=== AMAZON ML CHALLENGE 2026: PRO FEATURE ENGINEERING ===
1. Loading candidate pairs...
Total candidate pairs: 65,993,761

2. Loading raw sources...

3. Normalizing business names...

4. Merging clean text data onto pairs...

5. Attaching Ground Truth labels...


KeyError: 'source2_entity_id'